# Verify one ZM channel — start with FIR only

Run one cell at a time and stop at **Stop here: review FIR**. No AcqKnowledge comparison has been run yet.

For this first test, save a **separate copy** of one recording with its unchanged **raw ZM** and the **FIR result**. Use FIR 28–500 Hz, Blackman −61 dB, optimized coefficient count, and the entire waveform. Do not crop, resample or change units. Comb and RMS channels are not needed yet.

We compare the original raw samples with the raw channel retained in that copy, then compare Python FIR against AcqKnowledge FIR, sample for sample. Sample zero must represent the same instant. Errors use the original channel's units; sample indices are zero-based.

The function below is a standalone snapshot from `notebook.ipynb`; its TODOs remain unresolved. No acceptance threshold is assumed and no files are written. Future comb/RMS cells remain below the FIR stopping point.

In [ ]:
from pathlib import Path
import bioread
import numpy as np
import pandas as pd
from scipy import signal

## 1. Read the original and AcqKnowledge reference
Replace the two paths, then run the loading cell.

In [ ]:
raw_file = Path("reference/original.acq")
reference_file = Path("reference/zm_cleaning_stages.acq")

In [ ]:
raw_recording = bioread.read(raw_file)
reference_recording = bioread.read(reference_file)

In [ ]:
pd.DataFrame([
    {"file": label, "index": i, "name": ch.name, "units": ch.units,
     "rate_hz": ch.samples_per_second, "samples": len(ch.data)}
    for label, recording in [("original", raw_recording), ("reference", reference_recording)]
    for i, ch in enumerate(recording.channels)
])

Choose the **original raw ZM index**, then the **raw ZM and FIR indices in the reference file**. Use the Python `index` column above. Only these three choices are needed for FIR.

In [ ]:
raw_index = None
reference_indices = {"raw": None, "fir": None}

In [ ]:
raw_channel = raw_recording.channels[raw_index]
reference_channels = {stage: reference_recording.channels[index]
                      for stage, index in reference_indices.items()}
raw = np.array(raw_channel.data, dtype=float, copy=True)
sampling_rate = raw_channel.samples_per_second
reference = {stage: np.asarray(ch.data, dtype=float)
             for stage, ch in reference_channels.items()}
pd.DataFrame([
    {"stage": stage, "samples": len(ch.data), "rate_hz": ch.samples_per_second,
     "units": ch.units}
    for stage, ch in {"original": raw_channel, **reference_channels}.items()
])

Confirm equal sampling rates, sample counts and units above. The comparison stops on incompatible inputs rather than silently cropping or shifting them. Sample zero must represent the same time. These are the only structural guards; there are no automatic acceptance thresholds.

In [ ]:
for stage, ch in reference_channels.items():
    if ch.samples_per_second != sampling_rate or ch.units != raw_channel.units:
        raise ValueError(f"{stage}: sampling rate or units differ from the original.")

def compare(actual, expected):
    if actual.ndim != 1 or actual.shape != expected.shape or actual.size == 0:
        raise ValueError("Signals must be 1D with the same nonzero length.")
    if not (np.isfinite(actual).all() and np.isfinite(expected).all()):
        raise ValueError("Signals contain missing or infinite samples.")
    difference = actual - expected
    worst = int(np.argmax(np.abs(difference)))
    return pd.Series({
        "max_absolute_error": np.max(np.abs(difference)),
        "rmse": np.sqrt(np.mean(difference ** 2)),
        "mean_error": np.mean(difference),
        "worst_sample": worst,
        "worst_time_seconds": worst / sampling_rate,
    })

In [ ]:
raw_comparison = compare(raw, reference["raw"])
raw_comparison

**Raw-data checkpoint:** all errors should be zero for identical saved raw samples. The next cell stops if they differ. Investigate a wrong channel, changed scaling, different recording or altered raw data before changing the FIR function. A storage-related difference also needs investigation; do not bypass it silently.

In [ ]:
if not np.array_equal(raw, reference["raw"]):
    raise ValueError("Raw samples differ. Resolve the input mismatch before testing FIR.")

## 2. FIR — record the settings, then run the function

Record the Windows AcqKnowledge version and reported FIR coefficient count below, if available. The Python count is calculated for comparison; do not adjust it just to reduce an error without recording the reason.

In [ ]:
acqknowledge_version = ""  # Fill in from the software.
acqknowledge_fir_taps = None  # Fill in from its filter history if available.
python_fir_taps = int(4 * sampling_rate / 28)
if python_fir_taps % 2 == 0:
    python_fir_taps += 1
pd.Series({"AcqKnowledge version": acqknowledge_version,
           "AcqKnowledge coefficient count": acqknowledge_fir_taps,
           "Python coefficient count": python_fir_taps})

In [ ]:
def biopac_fir_bandpass(data, sampling_rate, low_hz=28.0, high_hz=500.0):
    """
    Approximates the specified AcqKnowledge FIR band-pass filter settings.
    Equivalence requires sample-by-sample numerical comparison with AcqKnowledge.

    Default parameters (28–500 Hz) reflect our protocol's settings.
    Returns a new array with the same length as the input.

    References:
        AcqKnowledge 5 Software Guide:
        - p. 334: Number of Coefficients formula and cutoff settings.
        - Appendix B, p. 720: Blackman -61 dB window parameter table.
    """
    # Blackman -61 dB weights from the BIOPAC manual.
    fir_window = [0.44959, 0.49364, 0.05677, 0.00000]
    fir_padding = "reflect"

    # BIOPAC formula: four times the sampling rate divided by the lowest cutoff.
    # TODO: verify integer rounding against AcqKnowledge's reported coefficient count.
    num_taps = int(4 * sampling_rate / low_hz)

    # Design choice: use an odd tap count for integer-sample centering.
    # BIOPAC documents this adjustment for Derivative Calculation.
    # TODO: verify whether FIR band-pass "Optimize" mode also applies it.
    if num_taps % 2 == 0:
        num_taps += 1

    # SciPy's firwin requires cutoff frequencies strictly below Nyquist.
    nyquist = sampling_rate / 2.0
    if high_hz >= nyquist:
        raise ValueError(
            f"high_hz ({high_hz} Hz) must be strictly less than the Nyquist "
            f"frequency ({nyquist} Hz) for sampling_rate={sampling_rate} Hz."
        )

    # Create the rectangular-window base; apply Blackman separately below.
    coefficients = signal.firwin(
        num_taps,
        [low_hz, high_hz],
        fs=sampling_rate,
        pass_zero=False,
        window="boxcar",
        scale=False,
    )

    # Apply the window to the filter coefficients.
    # TODO: verify window sampling by comparing filtered impulses in both programs.
    coefficients *= signal.windows.general_cosine(num_taps, fir_window)

    # Set filter gain to 1.0 at the passband center.
    # TODO: verify normalization using the interior amplitude of a filtered sine wave.
    center_freq = (low_hz + high_hz) / 2.0
    _, gain = signal.freqz(coefficients, worN=[center_freq], fs=sampling_rate)
    coefficients /= abs(gain[0])

    # Padding preserves output length. Centered convolution compensates
    # the odd-length filter's half-window delay.
    left = (num_taps - 1) // 2
    right = num_taps // 2

    # TODO: verify reflected boundaries by comparing filtered ramps in both programs.
    padded = np.pad(data, (left, right), mode=fir_padding)

    # TODO: compare filtered impulses sample by sample to verify timing.
    return signal.convolve(padded, coefficients, mode="valid")

In [ ]:
python_fir = biopac_fir_bandpass(raw, sampling_rate)

In [ ]:
fir_comparison = compare(python_fir, reference["fir"])
fir_comparison

### Read the FIR differences

`max_absolute_error` is the largest sample difference. `rmse` measures the overall error size; `mean_error` shows signed bias. `worst_sample` and `worst_time_seconds` locate the largest difference. Zero errors mean identical values for this recording. Nonzero errors need investigation; neither high correlation nor a visually similar trace establishes equivalence.

Separate the boundaries from the interior next. The Python FIR uses half its coefficient count on each side. That boundary width describes **our filter**, not a verified AcqKnowledge boundary. All samples remain in the overall comparison. Recordings too short to have an interior omit that row.

In [ ]:
edge_samples = min((python_fir_taps - 1) // 2, len(raw) // 2)
regions = {"beginning": (0, edge_samples),
           "interior": (edge_samples, len(raw) - edge_samples),
           "end": (len(raw) - edge_samples, len(raw))}
region_rows = []
for name, (left, right) in regions.items():
    if right > left:
        errors = python_fir[left:right] - reference["fir"][left:right]
        region_rows.append({"region": name, "start_sample": left,
                            "stop_sample_exclusive": right, "samples": right - left,
                            "max_absolute_error": np.max(np.abs(errors)),
                            "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(region_rows)

### Inspect one interval

Start with the first second. Change `start_sample` to inspect the interior or the worst mismatch, then rerun the table and plot cells. For example, use `max(0, int(fir_comparison["worst_sample"]) - 20)` around the worst mismatch, or `max(0, len(raw) - 200)` near the end. This only changes the displayed interval; it never shifts or truncates the signals used for the overall comparison.

In [ ]:
start_sample = 0
samples_to_show = int(sampling_rate)
if not 0 <= start_sample < len(raw) or samples_to_show < 1:
    raise ValueError("Choose a valid start sample and a positive interval length.")
stop_sample = min(start_sample + samples_to_show, len(raw))
indices = np.arange(start_sample, stop_sample)
fir_interval = pd.DataFrame({
    "sample": indices,
    "time_seconds": indices / sampling_rate,
    "python": python_fir[start_sample:stop_sample],
    "acqknowledge": reference["fir"][start_sample:stop_sample],
})
fir_interval["difference"] = fir_interval["python"] - fir_interval["acqknowledge"]
fir_interval.head(20).style.format({"time_seconds": "{:.9g}", "python": "{:.17g}",
                                   "acqknowledge": "{:.17g}", "difference": "{:.17g}"})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(fir_interval["time_seconds"], fir_interval["acqknowledge"],
             label="AcqKnowledge", color="tab:blue")
axes[0].plot(fir_interval["time_seconds"], fir_interval["python"],
             label="Python", color="tab:orange", linestyle="--")
axes[0].set_ylabel(f"Amplitude ({raw_channel.units})")
axes[0].set_title("FIR comparison — selected interval")
axes[0].legend()
axes[1].plot(fir_interval["time_seconds"], fir_interval["difference"], color="tab:purple")
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel(f"Python − AcqKnowledge ({raw_channel.units})")
axes[1].set_xlabel("Time from recording start (seconds)")
fig.tight_layout()
plt.show()

## Stop here: review FIR

Do not run comb or RMS yet. Record the measured errors and what they show before changing a function. No tolerance has been justified yet.

| Difference pattern | Next focused test |
| --- | --- |
| Mostly at the boundaries | Compare boundary extension using the same ramp or nonzero-start signal in both programs. |
| A shifted waveform | Filter a centered impulse in both programs and compare its position and shape. |
| Consistent amplitude difference | Filter the same 264 Hz sine and compare amplitude away from the boundaries. |
| Different shape throughout | Check reported tap count, then compare impulse responses to investigate coefficients/window sampling. |

These are diagnostic clues, not proof of a cause. Use the **same samples and sampling rate** for any controlled test. An impulse tests the filter response; a sine tests gain; a ramp helps distinguish boundary treatment. Only create these additional tests if the first real recording shows a mismatch that needs them.

Agreement for this recording does not establish every possible input case. Keep the native reference and note the software version, settings and observed errors. Change one assumption at a time.

**FIR findings:** pending real reference data.

---
The remaining cells are for later, after we review FIR.

## 3. COMB — continue after reviewing FIR

The reference must retain **FIR and comb results for the same channel**. AcqKnowledge settings: 60 Hz, Q=5, all harmonics up to Nyquist, entire waveform.

First test Python comb on **AcqKnowledge FIR**. This isolates the comb function. Then compare the complete Python sequence to see any error inherited from FIR. The sample table, regional errors and plot below use the **isolated** result.

In [ ]:
comb_index = None  # Reference file's comb result; set only when ready for this step.
reference_channels["comb"] = reference_recording.channels[comb_index]
ch = reference_channels["comb"]
if ch.samples_per_second != sampling_rate or ch.units != raw_channel.units:
    raise ValueError("Comb sampling rate or units differ from raw.")
reference["comb"] = np.asarray(ch.data, dtype=float)
if reference["comb"].shape != reference["fir"].shape or reference["comb"].size == 0:
    raise ValueError("COMB: input and output must have equal nonzero lengths.")
if not np.isfinite(reference["comb"]).all():
    raise ValueError("COMB: reference contains missing or infinite samples.")
pd.DataFrame([{"stage": name, "samples": len(reference[name]),
               "rate_hz": reference_channels[name].samples_per_second,
               "units": reference_channels[name].units}
              for name in ("fir", "comb")])

In [ ]:
def biopac_comb_band_stop(data, sampling_rate, base_hz=60.0, q=5.0):
    """
    Approximates the specified AcqKnowledge Comb Band Stop filter settings.
    Equivalence requires sample-by-sample numerical comparison with AcqKnowledge.

    Apply to the band-pass filtered signal. Cascades IIR notch filters at
    base_hz and its integer multiples strictly below Nyquist.

    Each notch uses the same Q, with bandwidth = center_frequency / Q.
    Returns a new array with the same length as the input.

    Reference:
        AcqKnowledge 5 Software Guide, pp. 337–338: Comb Band Stop.
    """
    nyquist = sampling_rate / 2.0

    if base_hz <= 0 or base_hz >= nyquist:
        raise ValueError(
            f"base_hz ({base_hz} Hz) must be positive and strictly less than "
            f"the Nyquist frequency ({nyquist} Hz)."
        )
    if q <= 0:
        raise ValueError(f"q ({q}) must be positive.")

    # Determine the highest harmonic below Nyquist.
    k_max = int(np.floor(nyquist / base_hz))

    # Exclude a harmonic that falls at or above Nyquist.
    if k_max * base_hz >= nyquist:
        k_max -= 1

    filtered = np.array(data, dtype=float, copy=True)

    # Apply notch filters in ascending frequency order.
    # TODO: verify numerical agreement with AcqKnowledge.
    for k in range(1, k_max + 1):
        frequency = k * base_hz

        # BIOPAC documents a cascade of IIR band-stop filters.
        # SciPy uses second-order notches with 3 numerator and 3 denominator coefficients.
        # TODO: verify coefficients against AcqKnowledge using an impulse response.
        b, a = signal.iirnotch(frequency, q, fs=sampling_rate)

        # Apply one forward pass with zero initial state.
        # TODO: verify AcqKnowledge's processing direction and initial conditions.
        # Forward-backward filtering would also change the magnitude response.
        filtered = signal.lfilter(b, a, filtered)

    return filtered

In [ ]:
isolated_comb = biopac_comb_band_stop(reference["fir"], sampling_rate)

In [ ]:
comb_comparison = compare(isolated_comb, reference["comb"])
comb_comparison

Compare the complete sequence separately. If the isolated result agrees but this one differs, an earlier step contributes to the mismatch.

In [ ]:
python_comb = biopac_comb_band_stop(python_fir, sampling_rate)
compare(python_comb, reference["comb"])

### Inspect regions of the isolated result

For comb, the first/last second are diagnostic intervals only: an IIR filter has no fixed finite boundary width, and transients can last longer. For RMS, separate the first 999 startup samples from complete windows, and inspect the end separately. No region is excluded from the overall comparison.

In [ ]:
edge_samples = min(int(sampling_rate), len(raw) // 2)
regions = {"beginning": (0, edge_samples),
           "interior": (edge_samples, len(raw) - edge_samples),
           "end": (len(raw) - edge_samples, len(raw))}
region_rows = []
for name, (left, right) in regions.items():
    if right > left:
        errors = isolated_comb[left:right] - reference["comb"][left:right]
        region_rows.append({"region": name, "start_sample": left,
                            "stop_sample_exclusive": right, "samples": right - left,
                            "max_absolute_error": np.max(np.abs(errors)),
                            "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(region_rows)

### Inspect one COMB interval

Change `start_sample`, then rerun the table and plot. To inspect the worst mismatch use `max(0, int(comb_comparison["worst_sample"]) - 20)`; near the end use `max(0, len(raw) - 200)`. All indices are zero-based; neither array is shifted. This selection affects only the display.

In [ ]:
start_sample = 0
samples_to_show = int(sampling_rate)
if not 0 <= start_sample < len(raw) or samples_to_show < 1:
    raise ValueError("Choose a valid start sample and a positive interval length.")
stop_sample = min(start_sample + samples_to_show, len(raw))
indices = np.arange(start_sample, stop_sample)
comb_interval = pd.DataFrame({
    "sample": indices, "time_seconds": indices / sampling_rate,
    "python": isolated_comb[start_sample:stop_sample],
    "acqknowledge": reference["comb"][start_sample:stop_sample],
})
comb_interval["difference"] = comb_interval["python"] - comb_interval["acqknowledge"]
comb_interval.head(20).style.format({"time_seconds": "{:.9g}", "python": "{:.17g}",
                                     "acqknowledge": "{:.17g}", "difference": "{:.17g}"})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(comb_interval["time_seconds"], comb_interval["acqknowledge"],
             label="AcqKnowledge", color="tab:blue")
axes[0].plot(comb_interval["time_seconds"], comb_interval["python"],
             label="Python on native input", color="tab:orange", linestyle="--")
axes[0].set_ylabel(f"Amplitude ({raw_channel.units})")
axes[0].set_title("COMB comparison — isolated function, selected interval")
axes[0].legend()
axes[1].plot(comb_interval["time_seconds"], comb_interval["difference"], color="tab:purple")
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel(f"Python − AcqKnowledge ({raw_channel.units})")
axes[1].set_xlabel("Time from recording start (seconds)")
fig.tight_layout()
plt.show()

## Stop here: review comb

- Mostly startup error: investigate initial conditions using the same nonzero-start input.
- Timing or waveform differences throughout: compare an impulse response to investigate coefficients and processing direction.
- Frequency-dependent differences: compare the same sine signals near 60 Hz and its harmonics to investigate notch width and gain.

These are clues, not proof. Inspect the isolated result before attributing a difference to comb. Record observations; do not change several assumptions at once or widen a tolerance to hide the mismatch.

**Comb findings:** pending real reference data. Continue to RMS only after reviewing this step.

## 4. RMS — continue after reviewing comb

The reference must retain **comb and RMS results for the same channel**. AcqKnowledge settings: Average over samples, 1000 samples, Root mean square, Remove baseline unchecked, entire waveform.

First test Python RMS on **AcqKnowledge comb**. Then compare the complete Python sequence. The table and plot below use the **isolated** result. The fixed denominator of 999 is still provisional.

In [ ]:
rms_index = None  # Reference file's RMS result; set only when ready for this step.
reference_channels["rms"] = reference_recording.channels[rms_index]
ch = reference_channels["rms"]
if ch.samples_per_second != sampling_rate or ch.units != raw_channel.units:
    raise ValueError("RMS sampling rate or units differ from raw.")
reference["rms"] = np.asarray(ch.data, dtype=float)
if reference["rms"].shape != reference["comb"].shape or reference["rms"].size == 0:
    raise ValueError("RMS: input and output must have equal nonzero lengths.")
if not np.isfinite(reference["rms"]).all():
    raise ValueError("RMS: reference contains missing or infinite samples.")
pd.DataFrame([{"stage": name, "samples": len(reference[name]),
               "rate_hz": reference_channels[name].samples_per_second,
               "units": reference_channels[name].units}
              for name in ("comb", "rms")])

In [ ]:
def biopac_moving_rms(data, window_samples=1000):
    """
    Approximates AcqKnowledge Transform > Integrate > Root mean square.

    Uses a trailing window of 1000 samples by default, matching our protocol.
    Each output includes the current sample and preceding samples.
    At the beginning, only available samples contribute to the sum.

    Uses a provisional fixed denominator of window_samples - 1,
    including during startup. AcqKnowledge's manual gives conflicting
    descriptions of the denominator.

    Expects a nonempty 1D signal.
    Returns a new array with the same length, leaving the input unchanged.
    Exact AcqKnowledge equivalence requires numerical comparison.
    """
    if window_samples < 2:
        raise ValueError("window_samples must be at least 2.")

    squared = np.asarray(data, dtype=float) ** 2

    # Trailing sums: current sample plus up to window_samples - 1 previous samples.
    # Taking the first len(data) outputs preserves timing and signal length.
    # No reflected samples are introduced at the beginning.
    window_sums = np.convolve(
        squared, np.ones(window_samples), mode="full"
    )[:len(squared)]

    # Provisional fixed denominator, including for incomplete startup windows.
    # TODO: verify both startup values and s versus s-1 against AcqKnowledge.
    # For a full 1000-sample window of alternating +1/-1:
    # denominator 1000 -> 1.000000; denominator 999 -> 1.000500375...
    mean_squared = window_sums / (window_samples - 1)

    return np.sqrt(mean_squared)

In [ ]:
isolated_rms = biopac_moving_rms(reference["comb"])

In [ ]:
rms_comparison = compare(isolated_rms, reference["rms"])
rms_comparison

Compare the complete sequence separately. If the isolated result agrees but this one differs, an earlier step contributes to the mismatch.

In [ ]:
python_rms = biopac_moving_rms(python_comb)
compare(python_rms, reference["rms"])

### Inspect regions of the isolated result

For comb, the first/last second are diagnostic intervals only: an IIR filter has no fixed finite boundary width, and transients can last longer. For RMS, separate the first 999 startup samples from complete windows, and inspect the end separately. No region is excluded from the overall comparison.

In [ ]:
startup_end = min(999, len(raw))
end_start = max(startup_end, len(raw) - 1000)
regions = {"startup": (0, startup_end),
           "complete_windows_interior": (startup_end, end_start),
           "complete_windows_end": (end_start, len(raw))}
region_rows = []
for name, (left, right) in regions.items():
    if right > left:
        errors = isolated_rms[left:right] - reference["rms"][left:right]
        region_rows.append({"region": name, "start_sample": left,
                            "stop_sample_exclusive": right, "samples": right - left,
                            "max_absolute_error": np.max(np.abs(errors)),
                            "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(region_rows)

### Inspect one RMS interval

Change `start_sample`, then rerun the table and plot. To inspect the worst mismatch use `max(0, int(rms_comparison["worst_sample"]) - 20)`; near the end use `max(0, len(raw) - 200)`. All indices are zero-based; neither array is shifted. This selection affects only the display.

In [ ]:
start_sample = 0
samples_to_show = int(sampling_rate)
if not 0 <= start_sample < len(raw) or samples_to_show < 1:
    raise ValueError("Choose a valid start sample and a positive interval length.")
stop_sample = min(start_sample + samples_to_show, len(raw))
indices = np.arange(start_sample, stop_sample)
rms_interval = pd.DataFrame({
    "sample": indices, "time_seconds": indices / sampling_rate,
    "python": isolated_rms[start_sample:stop_sample],
    "acqknowledge": reference["rms"][start_sample:stop_sample],
})
rms_interval["difference"] = rms_interval["python"] - rms_interval["acqknowledge"]
rms_interval.head(20).style.format({"time_seconds": "{:.9g}", "python": "{:.17g}",
                                     "acqknowledge": "{:.17g}", "difference": "{:.17g}"})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(rms_interval["time_seconds"], rms_interval["acqknowledge"],
             label="AcqKnowledge", color="tab:blue")
axes[0].plot(rms_interval["time_seconds"], rms_interval["python"],
             label="Python on native input", color="tab:orange", linestyle="--")
axes[0].set_ylabel(f"Amplitude ({raw_channel.units})")
axes[0].set_title("RMS comparison — isolated function, selected interval")
axes[0].legend()
axes[1].plot(rms_interval["time_seconds"], rms_interval["difference"], color="tab:purple")
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel(f"Python − AcqKnowledge ({raw_channel.units})")
axes[1].set_xlabel("Time from recording start (seconds)")
fig.tight_layout()
plt.show()

### Compare the two fixed RMS denominators

Use the **same native comb input**, trailing window and 1000-sample width for both candidates. Only the denominator changes: 1000 versus 999. Compare complete windows separately from startup. The first complete trailing window ends at index **999**.

A smaller error is evidence, not an automatic choice: alignment and startup rules must also agree. This tests two **fixed** denominators, not an adaptive startup denominator. The cleaning function remains unchanged.

In [ ]:
window_sums = np.convolve(reference["comb"] ** 2, np.ones(1000), mode="full")[:len(raw)]
denominator_rows = []
for denominator in (1000, 999):
    candidate = np.sqrt(window_sums / denominator)
    for region, left, right in [("startup", 0, min(999, len(raw))),
                                ("complete_windows", 999, len(raw))]:
        if right > left:
            errors = candidate[left:right] - reference["rms"][left:right]
            denominator_rows.append({"denominator": denominator, "region": region,
                                     "samples": right - left,
                                     "max_absolute_error": np.max(np.abs(errors)),
                                     "rmse": np.sqrt(np.mean(errors ** 2))})
pd.DataFrame(denominator_rows)

## Stop here: review RMS

- Differences only in the first 999 samples: investigate startup handling.
- A shifted envelope: investigate trailing versus centered alignment with a single impulse at the same position in both programs.
- A small consistent amplitude difference in complete windows: inspect the two denominator candidates above.

If needed, apply **only RMS** to an identical sequence of alternating +1/-1 samples in both programs. A complete 1000-sample window gives 1.0 with denominator 1000, or approximately 1.000500375 with denominator 999. Read saved values at full precision and inspect startup separately. Filtering that test signal first would change the input and invalidate these expected values.

**RMS findings:** pending real reference data. No automatic PASS threshold has been assigned. Agreement on one recording does not prove every input or boundary case.

The isolated comparison feeds the **AcqKnowledge preceding stage** into the Python function. If that agrees but the complete sequence differs, an earlier step contributes to the mismatch. Agreement on one recording supports that recording only; it does not prove every boundary case.

## 5. Inspect individual samples
Choose a stage and sample interval. Inspect the beginning, around a burst, near the reported worst sample, and the end. Do not shift signals to make them agree: timing is part of this test. For RMS, the first 999 samples deserve separate attention because startup and the denominator are unresolved.

In [ ]:
stage = "fir"
start = 0
stop = 20
python_stages = {"fir": python_fir, "comb": python_comb, "rms": python_rms}
actual = python_stages[stage]
expected = reference[stage]
pd.DataFrame({
    "sample": np.arange(len(actual))[start:stop],
    "python": actual[start:stop],
    "acqknowledge": expected[start:stop],
    "difference": (actual - expected)[start:stop],
})

## Record what needs changing

- AcqKnowledge version: …
- FIR reported coefficient count: …
- FIR: rounding/odd count, window sampling, gain and boundaries remain to verify.
- Comb: coefficients, processing direction and initial conditions remain to verify.
- RMS: fixed 999 versus 1000 denominator, startup values and timing remain to verify.

Keep the reference `.acq` and these results. Change only the first disagreeing function, restart the kernel and rerun the comparisons. No files are written by this notebook.